In [11]:
import os
os.environ['OMOL_MODEL_PATH'] = '/Users/mgt16/fairchem_models/esen_sm_conserving_all.pt'
from architector import build_complex, convert_io_molecule, CalcExecutor, view_structures

In [105]:
from architector.io_ptable import ligands_dict
from ase import units
import numpy as np

In [13]:
mol = convert_io_molecule('CCCCCC')
mol.get_dists(atom_type_pairs=[['C','C']])

,atom_pair,bond_type,smiles,smiles_index,distance,sum_cov_radii,atom_symbols
1,"(0, 1)",explicit_bond,CCCCCC,None,1.520050,1.5,C-C
2,"(0, 2)",implicit_bond,CCCCCC,None,2.482209,1.5,C-C
3,"(0, 3)",implicit_bond,CCCCCC,None,3.041699,1.5,C-C
4,"(0, 4)",implicit_bond,CCCCCC,None,3.675875,1.5,C-C
5,"(0, 5)",implicit_bond,CCCCCC,None,4.719581,1.5,C-C
8,"(1, 2)",explicit_bond,CCCCCC,None,1.520038,1.5,C-C
9,"(1, 3)",implicit_bond,CCCCCC,None,2.557695,1.5,C-C
10,"(1, 4)",implicit_bond,CCCCCC,None,3.065700,1.5,C-C
11,"(1, 5)",implicit_bond,CCCCCC,None,3.682086,1.5,C-C
15,"(2, 3)",explicit_bond,CCCCCC,None,1.537216,1.5,C-C


In [38]:
inputDict = {'core':{'metal':'Fe','coreCN':6}, #Specify metal coordination number (CN)
            'ligands':['bipy']*3, # Specify what is filling the coordination environment
            'parameters':{'full_method':'UFF'}} # No additional parameters needed for default
out1 = build_complex(inputDict) # Now just build using the dictionary!

/Users/mgt16/software/omol_2/Architector/architector/io_core.py:69: RuntimeWarning: invalid value encountered in arccos
  np.arccos(


In [113]:
out1['octahedral_0_nunpairedes_4_charge_2']['metal_center_symmetry']

'octahedral'

In [39]:
mol = convert_io_molecule(out1[list(out1.keys())[0]]['mol2string'])

In [40]:
mol.get_dists()

,atom_pair,bond_type,smiles,smiles_index,distance,sum_cov_radii,atom_symbols
0,"(0, 1)",explicit_bond,c1ccc(nc1)c1ccccn1,4,1.984862,1.87,Fe-N
1,"(0, 12)",explicit_bond,c1ccc(nc1)c1ccccn1,11,1.975712,1.87,Fe-N
2,"(0, 21)",explicit_bond,c1ccc(nc1)c1ccccn1,4,1.978988,1.87,Fe-N
3,"(0, 32)",explicit_bond,c1ccc(nc1)c1ccccn1,11,1.978853,1.87,Fe-N
4,"(0, 41)",explicit_bond,c1ccc(nc1)c1ccccn1,4,1.979125,1.87,Fe-N
5,"(0, 52)",explicit_bond,c1ccc(nc1)c1ccccn1,11,1.979061,1.87,Fe-N


In [41]:
view_structures(mol, vis_distances=True)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [42]:
len(ligands_dict)

146

In [43]:
allowable_denticites=[1,2,3,6]
possible_ligs = []
for key,val in ligands_dict.items():
    ligcharge = convert_io_molecule(val['smiles']).charge
    # print(ligcharge)
    if (len(val['coordList']) in allowable_denticites) and (
        ((ligcharge * int(6/len(val['coordList']))) + 2) > -3
    ) and ('edge' not in val.get('ligType','')) and (
        ('sandwich' not in val.get('ligType',''))
    ):
        possible_ligs.append(val)

In [44]:
len(possible_ligs)

88

In [46]:
possible_ligs[0:3]

[{'smiles': 'O', 'coordList': [0], 'ligType': 'mono', 'ligCharge': 0},
 {'smiles': 'OO', 'coordList': [0], 'ligType': 'mono'},
 {'smiles': 'O=O', 'coordList': [0], 'ligType': 'mono'}]

In [116]:
def eval_lig(lig, task_dict={
    'sco':True,
    'solvents':['water','acetone','octanol','hexane'],
    },
    debug=True):
    """eval_lig
    """
    good = True
    error = ''
    output_dict = dict()
    if task_dict.get('sco', False):
        arch_out = dict()
        nlig = int(6/len(lig['coordList']))
        architector_input = {
            'core':{'metal':'Fe','coreType':'octahedral'},
            'ligands':[lig]*nlig,
            'parameters':{
                'metal_ox':2,
                'metal_spin':4,
                'assemble_method':'UFF',
                'full_method':'UFF'}
        }
        arch_mol = None
        try:
            if debug:
                print('Building Complex')
            arch_out = build_complex(architector_input)
            arch_mol = convert_io_molecule(arch_out[list(arch_out.keys())[0]]['mol2string'])
            output_dict['architector_uff_mol2'] = arch_out[list(arch_out.keys())[0]]['mol2string']
        except Exception as e:
            good = False
            error += e
        if len(arch_out) == 0:
            good = False
            error += 'Architector produced no output'
        else:
            mol = convert_io_molecule(arch_mol)
            mol.uhf = 0
            if debug:
                print('Relaxing LS Complex')
            low_spin = CalcExecutor(mol, method='omol', relax=True, fmax=0.05)
            if low_spin.successful:
                output_dict['low_spin_mol2_omol'] = low_spin.mol.write_mol2('ls',writestring=True)
            else:
                error += 'low spin omol failed'
            if debug:
                print('Relaxing HS Complex')
            mol = convert_io_molecule(arch_mol)
            high_spin = CalcExecutor(mol, method='omol', relax=True, fmax=0.05)
            if high_spin.successful:
                output_dict['high_spin_mol2_omol'] = high_spin.mol.write_mol2('hs',writestring=True)
            else:
                error += 'high spin omol failed'
            if low_spin.successful and high_spin.successful:
                output_dict['sco_kcal'] = (high_spin.energy - low_spin.energy) / (units.kcal/units.mol)
    if good and len(task_dict.get('solvents',[])) > 0:
        for solv in task_dict['solvents']:
            if debug:
                print('Evaluating {} solvent'.format(solv))
            mol = CalcExecutor(arch_mol,
                method='GFN2-xTB', xtb_solvent=solv,
                relax=False, store_results=True)
            if mol.successful:
                output_dict['{}_gsolv_eV'.format(solv)] = mol.results['gsolv_eV']
                output_dict['{}_hl_gap_eV'.format(solv)] = mol.results['hl_gap_eV']
                output_dict['{}_dipole'.format(solv)] = np.linalg.norm(mol.results['dipole'])
            else:
                error += '{} solvent failed XTB'.format(solv)
    if debug:
        print('Done')
    output_dict['error'] = error
    return output_dict


In [115]:
possible_ligs[20]

{'smiles': 'NCCN', 'coordList': [0, 3], 'ligType': 'bi_cis'}

In [110]:
thing = eval_lig(possible_ligs[20])

Building Complex


/Users/mgt16/software/omol_2/Architector/architector/io_lig.py:793: RuntimeWarning: invalid value encountered in sqrt
  L[i, i] = np.sqrt(np.max(l[natoms - 1 - i], 0))
/Users/mgt16/software/omol_2/Architector/architector/io_core.py:69: RuntimeWarning: invalid value encountered in arccos
  np.arccos(


Relaxing LS Complex
Relaxing HS Complex
Evaluating water solvent
Evaluating acetone solvent
Evaluating octanol solvent
Evaluating hexane solvent
Done


In [111]:
thing

{'low_spin_mol2_omol': '@<TRIPOS>MOLECULE\nls Charge: 2 Unpaired_Electrons: 0 XTB_Unpaired_Electrons: 0 XTB_Charge: 2\n    37    39     1     0     0\nSMALL\nNoCharges\n****\nGenerated from Architector\n\n@<TRIPOS>ATOM\n     1 Fe1      -0.0004    0.0475   -0.0449   Fe        1 RES1   0.0000\n     2 N1        0.0128   -0.0882    2.0304   N.4       1 RES1   0.0000\n     3 C1        0.3389   -1.4872    2.4217   C.3       1 RES1   0.0000\n     4 C2       -0.3733   -2.4213    1.4725   C.3       1 RES1   0.0000\n     5 N2       -0.0138   -2.0287    0.0827   N.4       1 RES1   0.0000\n     6 H1        0.6561    0.5359    2.5107   H         1 RES1   0.0000\n     7 H2       -0.8988    0.1431    2.4206   H         1 RES1   0.0000\n     8 H3        0.0662   -1.6971    3.4561   H         1 RES1   0.0000\n     9 H4        1.4182   -1.6255    2.3367   H         1 RES1   0.0000\n    10 H5       -1.4537   -2.3216    1.5902   H         1 RES1   0.0000\n    11 H6       -0.1203   -3.4606    1.6831   H   

In [75]:
view_structures(thing['low_spin_mol2_omol'],vis_distances=True)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [103]:
view_structures(thing['high_spin_mol2_omol'],vis_distances=True)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [104]:
mol.results.keys()

dict_keys(['sas', 'born_radii', 'gsolv_eV', 'hl_gap_eV', 'energy', 'forces', 'charges', 'coveCNs', 'dipole'])

In [102]:
mol.mol.get_rad_gyration()

np.float64(2.269138123355023)